# Capstone #4 — MCP-Powered Personal Assistant

*Notebook 29*

Put it all together. Connect to filesystem, web, and time MCP servers to build a real-world personal assistant.
<br>
<br>

**Topics:**

- Three MCP servers working together

- Server-level approval for write and delete operations

- Saving results is a side effect — handle it with the same care as write actions

- `REASONING_MODEL` for decision-making

- Real-world tasks: research, summarize, save results

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

print(f"✅ Setup complete")
print(f"   Assistant model: {REASONING_MODEL}")
print(f"   Servers: filesystem + web fetch + time")

---

## 🏗️ System Architecture

The personal assistant combines three MCP servers:

```
User Request
      ↓
Personal Assistant (REASONING_MODEL)
   ├── Filesystem MCP  → read/write local files
   │                    (write/delete require approval)
   ├── Web Fetch MCP   → retrieve web content
   └── Time MCP        → current time, timezone conversion
```

the agent makes one tool call at a time; using a stronger model improves those choices on coordination-heavy tasks.

---

## 📁 Phase 1: Prepare the Workspace

In [ ]:
workspace = Path("assistant_workspace")
workspace.mkdir(exist_ok=True)

(workspace / "reading_list.txt").write_text("""Reading List
============
Books to read:
- The Pragmatic Programmer
- Clean Code by Robert Martin
- Designing Data-Intensive Applications

Articles to read:
- OpenAI Agents SDK documentation
- Model Context Protocol specification
""")

(workspace / "project_notes.txt").write_text("""Project: AI Agents Course
=========================
Status: In progress
Modules completed: 1-5
Next: Record video lessons
Deadline: End of month

Key topics covered:
- Agent fundamentals
- Multi-agent systems
- Production patterns
- MCP integration
""")

# --------------------------------------------------------------
print(f"✅ Workspace ready: {workspace.resolve()}")
print(f"   Files: {[f.name for f in workspace.iterdir()]}")

---

## 🤖 Phase 2: Build the Assistant

Three MCP servers, write approval on filesystem, `REASONING_MODEL` for decision-making.

In [ ]:
print("🤖 PERSONAL ASSISTANT DEMO")
print("=" * 60)

# Server configurations (reused across tasks)
fs_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
}
fs_approval = {
    "always": {"tool_names": ["write_file", "create_directory", "move_file", "delete_file"]},
    "never": {"tool_names": ["read_file", "list_directory", "get_file_info", "search_files"]}
}
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}
time_params = {"command": "uvx", "args": ["mcp-server-time"]}

assistant_instructions = (
    "You are a personal productivity assistant with access to:\n"
    "- Filesystem tools: read and write files in the workspace\n"
    "- Web fetch: retrieve content from URLs\n"
    "- Time tools: get current time and convert between timezones\n\n"
    "Be proactive about combining tools — research on the web, then save results locally.\n"
    "Use the reasoning model's strengths: plan multi-step tasks before executing them."
)

# -------------------------------------------------------
# Task 1: Use time + filesystem together
# -------------------------------------------------------
print("\n📋 Task 1: Time-stamped daily summary")
print("-" * 40)

async with (
    MCPServerStdio(name="Filesystem", params=fs_params, cache_tools_list=True, require_approval=fs_approval) as fs_server,
    MCPServerStdio(name="WebFetch", params=fetch_params, cache_tools_list=True) as fetch_server,
    MCPServerStdio(name="Time", params=time_params, cache_tools_list=True) as time_server
):
    assistant = Agent(
        name="PersonalAssistant",
        instructions=assistant_instructions,
        model=REASONING_MODEL,
        mcp_servers=[fs_server, fetch_server, time_server]
    )

    result = await Runner.run(assistant, input="What time is it now? Read my project_notes.txt and reading_list.txt, then create a daily_summary.txt with today's date and a brief status of both.")

    # Handle any write approvals
    while result.interruptions:
        state = result.to_state()
        for interruption in state.get_interruptions():
            print(f"\n✍️  Write approval needed: {interruption.raw_item.name}")
            print(f"   Arguments: {interruption.raw_item.arguments}")
            decision = input("   Approve? (yes/no): ").strip().lower()
            if decision in ["yes", "y"]:
                state.approve(interruption)
            else:
                state.reject(interruption, message="User declined this write operation.")

        result = await Runner.run(assistant, state)

    print(result.final_output[:400])

> Saving results is still a write — `write_file` triggers the same approval flow as any other write action. A "harmless" save shouldn't get an exemption.

#### Task 2: Web Research + Save

Now let's combine web fetch with a file write — the approval flow is the same pattern we just used for Task 1.

In [ ]:
# Task 2: Web research + save (write approval required — same flow as Task 1)
print("\n📋 Task 2: Research and save")
print("-" * 40)

async with (
    MCPServerStdio(name="Filesystem", params=fs_params, cache_tools_list=True, require_approval=fs_approval) as fs_server,
    MCPServerStdio(name="WebFetch", params=fetch_params, cache_tools_list=True) as fetch_server,
    MCPServerStdio(name="Time", params=time_params, cache_tools_list=True) as time_server
):
    assistant = Agent(
        name="PersonalAssistant",
        instructions=assistant_instructions,
        model=REASONING_MODEL,
        mcp_servers=[fs_server, fetch_server, time_server]
    )

    result = await Runner.run(assistant, input="Fetch https://modelcontextprotocol.io and save a 3-sentence summary to mcp_research.txt")

    while result.interruptions:
        state = result.to_state()
        for interruption in state.get_interruptions():
            print(f"\n✍️  Write approval needed: {interruption.raw_item.name}")
            print(f"   Arguments: {interruption.raw_item.arguments}")
            decision = input("   Approve? (yes/no): ").strip().lower()
            if decision in ["yes", "y"]:
                state.approve(interruption)
            else:
                state.reject(interruption, message="User declined this write operation.")

        result = await Runner.run(assistant, state)

    print(result.final_output[:400])

print("\n" + "=" * 60)

# Verify files were created
for fname in ["daily_summary.txt", "mcp_research.txt"]:
    fpath = workspace / fname
    if fpath.exists():
        print(f"✅ {fname} created ({len(fpath.read_text())} chars)")

---

#### Cleanup

In [ ]:
import shutil
shutil.rmtree(workspace, ignore_errors=True)
print("✅ Workspace cleaned up")

---

## 💪 Practice Exercise

### Swap in a Different MCP Server

*Covers: MCP server substitution — swapping implementations*

Replace the Time server with a different MCP server of your choice and update the assistant to use it.

In [ ]:
# --------------------------------------------------------------
# 💪 Swap in a Different MCP Server
# --------------------------------------------------------------
# Objective: Replace the Time MCP server with a different one.
#
# Phases 1-2 (workspace + filesystem/fetch servers) are provided above.
# Your job: replace the third server and update the assistant.

# Option A: Memory server (note-taking with persistence)
# command: "npx", args: ["-y", "@modelcontextprotocol/server-memory"]

# Option B: Memory server — lets the agent store and retrieve persistent notes
# command: "npx", args: ["-y", "@modelcontextprotocol/server-memory"]

# Option C: Your own choice — browse https://github.com/modelcontextprotocol/servers

# -------------------------------------------------------
# Step 1: Create a new workspace
# -------------------------------------------------------
# TODO: Create exercise_workspace with some sample files

# -------------------------------------------------------
# Step 2: Connect three servers (provided pattern below)
# -------------------------------------------------------
# TODO: Replace the Time server params with your chosen server

# async with (
#      MCPServerStdio(name="Filesystem", params={...}, ...) as fs_server,
#      MCPServerStdio(name="WebFetch", params={...}, ...) as fetch_server,
#      MCPServerStdio(name="YourServer", params={...}, ...) as your_server  # NEW
# ):

# -------------------------------------------------------
# Step 3: Update the agent instructions
# -------------------------------------------------------
# TODO: Create an assistant that uses all three servers
#        Update instructions to describe what the new server does

# -------------------------------------------------------
# Step 4: Run a task that uses all three servers
# -------------------------------------------------------
# TODO: Design a task that exercises filesystem + fetch + your new server
#        Handle any write approvals using the pattern below:
#
# Example approval loop (reuse this pattern):
# while result.interruptions:
#     state = result.to_state()
#     for interruption in state.get_interruptions():
#         print(f"Approval needed: {interruption.raw_item.name}")
#         print(f"Arguments: {interruption.raw_item.arguments}")
#         decision = input("Approve? (yes/no): ").strip().lower()
#         if decision in ["yes", "y"]:
#             state.approve(interruption)
#         else:
#             state.reject(interruption, message="User rejected this operation.")
#     result = await Runner.run(assistant, state)

# -------------------------------------------------------
# Step 5: Cleanup
# -------------------------------------------------------
# TODO: Remove the exercise workspace

print("💡 Choose a server, implement the steps above, then run your assistant!")

---

## 🎯 Key Takeaways

**Three Servers, One Agent:**

- Any number of MCP servers can be combined in `mcp_servers=[...]`

- The agent sees all tools from all servers as one flat list

- Tools from different servers work together seamlessly
<br>
<br>

**Approval at the Server Level:**

- `require_approval` on `MCPServerStdio` protects destructive operations

- Per-tool policies: `{"always": {"tool_names": [...]}, "never": {"tool_names": [...]}}`

- Approval pauses the run — inspect interruptions, then approve or reject and resume
<br>
<br>

**REASONING_MODEL for Coordination:**

- Multi-server tasks benefit from stronger reasoning

- The model plans which server to use at each step

- Worth the higher cost for complex multi-step tasks

---

## 📍 Next Step

**Notebook 30: Project Structure & CLI**  

Move from notebooks to a structured Python project with a CLI entry point.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-29-capstone-4--mcp-assistant)

---